# EDA — Burnout & Cognitive Load Guardrail
## Sprint 4: Feature Engineering Exploratory Analysis

**Goal:** Validate the four raw features and the AFS composite before model training.

Sections:
1. Data loading (synthetic or warehouse)
2. Univariate distributions per feature
3. Correlation heatmap
4. AFS distribution and zone breakdown
5. Outlier inspection
6. Temporal trends

In [ ]:
import warnings

import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np
import pandas as pd

warnings.filterwarnings('ignore')
pd.set_option('display.float_format', '{:.4f}'.format)

from ml.training.dataset import temporal_split
from ml.training.features import AFS_COL, RAW_FEATURE_COLS, ZONE_COL, ResilienceZone
from ml.training.synthetic import SyntheticDataGenerator

SEED = 42
ZONE_COLORS = {
    ResilienceZone.GREEN.value:  '#2ECC71',
    ResilienceZone.YELLOW.value: '#F39C12',
    ResilienceZone.RED.value:    '#E74C3C',
}

## 1. Data Loading

Uses the synthetic generator for reproducibility.
Swap the cell below for a warehouse query when real data is available.

In [ ]:
gen = SyntheticDataGenerator(seed=SEED)
df = gen.generate_dataframe(n_teams=50, n_days=30)

print(f"Dataset shape: {df.shape}")
print(f"Teams: {df['team_id'].nunique()}")
print(f"Date range: {df['date_utc'].min()} → {df['date_utc'].max()}")
df.head()

In [ ]:
df.describe()

## 2. Univariate Distributions

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 8))
fig.suptitle('Raw Feature Distributions', fontsize=14, fontweight='bold')

feature_labels = {
    'calendar_density_score':     'Calendar Density Score',
    'after_hours_activity_index': 'After-Hours Activity Index',
    'context_switch_count':       'Context-Switch Count',
    'sprint_health_index':        'Sprint Health Index',
}

for ax, col in zip(axes.flat, RAW_FEATURE_COLS):
    zone_vals = {
        z: df.loc[df[ZONE_COL] == z, col]
        for z in [ResilienceZone.GREEN.value, ResilienceZone.YELLOW.value, ResilienceZone.RED.value]
    }
    for zone, vals in zone_vals.items():
        ax.hist(vals, bins=30, alpha=0.55, color=ZONE_COLORS[zone], label=zone, density=True)
    ax.set_title(feature_labels[col])
    ax.set_xlabel('Value [0, 1]')
    ax.set_ylabel('Density')
    ax.legend(fontsize=8)
    ax.set_xlim(0, 1)

plt.tight_layout()
plt.savefig('feature_distributions.png', dpi=120, bbox_inches='tight')
plt.show()
print('Saved: feature_distributions.png')

## 3. Correlation Heatmap

In [ ]:
corr = df[RAW_FEATURE_COLS + [AFS_COL]].corr()

fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(corr.values, vmin=-1, vmax=1, cmap='RdYlGn_r')
plt.colorbar(im, ax=ax, fraction=0.046)

labels = [c.replace('_', '\n') for c in corr.columns]
ax.set_xticks(range(len(labels)))
ax.set_yticks(range(len(labels)))
ax.set_xticklabels(labels, fontsize=8)
ax.set_yticklabels(labels, fontsize=8)

for i in range(len(corr)):
    for j in range(len(corr)):
        ax.text(j, i, f'{corr.values[i, j]:.2f}', ha='center', va='center', fontsize=8)

ax.set_title('Feature Correlation Matrix')
plt.tight_layout()
plt.savefig('correlation_heatmap.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nHigh correlations (|r| > 0.5, excluding diagonal):')
mask = (corr.abs() > 0.5) & (corr != 1.0)
pairs = [(r, c, corr.loc[r, c]) for r in corr.index for c in corr.columns if mask.loc[r, c] and r < c]
if pairs:
    for r, c, v in pairs:
        print(f'  {r} × {c}: {v:.3f}')
else:
    print('  None')

## 4. AFS Distribution and Zone Breakdown

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('Attention Fragmentation Score (AFS)', fontsize=14, fontweight='bold')

# Histogram with zone bands
for zone, color in ZONE_COLORS.items():
    vals = df.loc[df[ZONE_COL] == zone, AFS_COL]
    ax1.hist(vals, bins=25, alpha=0.6, color=color, label=zone)
ax1.axvline(40, color='orange', ls='--', lw=1.5, label='Yellow threshold')
ax1.axvline(70, color='red',    ls='--', lw=1.5, label='Red threshold')
ax1.set_xlabel('AFS Score [0, 100]')
ax1.set_ylabel('Count')
ax1.set_title('AFS Distribution')
ax1.legend(fontsize=8)

# Zone pie chart
zone_counts = df[ZONE_COL].value_counts()
ax2.pie(
    zone_counts.values,
    labels=zone_counts.index,
    colors=[ZONE_COLORS[z] for z in zone_counts.index],
    autopct='%1.1f%%',
    startangle=90,
)
ax2.set_title('Resilience Zone Distribution')

plt.tight_layout()
plt.savefig('afs_distribution.png', dpi=120, bbox_inches='tight')
plt.show()

print('\nAFS statistics:')
print(df[AFS_COL].describe().to_string())
print('\nZone counts:')
print(zone_counts.to_string())

## 5. Outlier Inspection

Flag records where any feature is > 3 standard deviations from the mean.

In [ ]:
z_scores = (df[RAW_FEATURE_COLS] - df[RAW_FEATURE_COLS].mean()) / df[RAW_FEATURE_COLS].std()
outlier_mask = (z_scores.abs() > 3).any(axis=1)

print(f'Outlier rows (any feature > 3σ): {outlier_mask.sum()} / {len(df)} ({outlier_mask.mean():.1%})')

fig, axes = plt.subplots(1, len(RAW_FEATURE_COLS), figsize=(14, 4))
for ax, col in zip(axes, RAW_FEATURE_COLS):
    df.boxplot(column=col, by=ZONE_COL, ax=ax, vert=True,
               boxprops=dict(color='navy'), medianprops=dict(color='red'))
    ax.set_title(col.replace('_', '\n'), fontsize=8)
    ax.set_xlabel('')
    plt.sca(ax)
    plt.xticks(fontsize=7)
fig.suptitle('Feature Box-Plots by Resilience Zone')
plt.tight_layout()
plt.savefig('boxplots.png', dpi=120, bbox_inches='tight')
plt.show()

if outlier_mask.sum() > 0:
    print('\nOutlier sample:')
    display(df[outlier_mask].head())

## 6. Temporal Trends

Mean AFS per day across all teams — check for temporal patterns or data drift.

In [ ]:
daily = df.groupby('date_utc')[AFS_COL].agg(['mean', 'std']).reset_index()
daily['date_utc'] = pd.to_datetime(daily['date_utc'])

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(daily['date_utc'], daily['mean'], color='steelblue', lw=2, label='Mean AFS')
ax.fill_between(
    daily['date_utc'],
    daily['mean'] - daily['std'],
    daily['mean'] + daily['std'],
    alpha=0.2, color='steelblue', label='±1 std'
)
ax.axhline(40, color='orange', ls='--', lw=1, label='Yellow threshold')
ax.axhline(70, color='red',    ls='--', lw=1, label='Red threshold')
ax.set_xlabel('Date')
ax.set_ylabel('AFS')
ax.set_title('Mean Daily AFS Across All Teams')
ax.legend(fontsize=9)
ax.yaxis.set_major_locator(mticker.MultipleLocator(10))
plt.tight_layout()
plt.savefig('temporal_afs.png', dpi=120, bbox_inches='tight')
plt.show()

## 7. Train / Val / Test Split Preview

In [ ]:
split = temporal_split(df)
print('Dataset split summary:')
for k, v in split.summary().items():
    print(f'  {k}: {v}')

# Zone distribution per partition
for name, y in [('train', split.y_train), ('val', split.y_val), ('test', split.y_test)]:
    unique, counts = np.unique(y, return_counts=True)
    zone_names = ['green', 'yellow', 'red']
    dist = {zone_names[int(k)]: int(v) for k, v in zip(unique, counts)}
    print(f'  {name} zone distribution: {dist}')